#**準備しておくこと**

指定されたフォルダへ音声ファイルをアップロードする

In [ ]:
# @title
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
student_id='00000001'#@param{type:"string"}
root='/content/drive/Shareddrives/専門演習I/前期前半/SpeechData'#@param{type:'string'}

In [ ]:
# @title
import os
import re
import wave
from collections import Counter

# チェック対象ディレクトリ（ColabではGoogle Driveにマウントして指定）
target_dir = os.path.join(root, student_id)

# 条件
MAX_DURATION = 5.0       # 最大5秒
REQUIRED_SR = 16000      # サンプリングレート
REQUIRED_CHANNELS = 1    # モノラル
EXPECTED_FILES = 20      # ファイル数

# ファイル名の正規表現パターン
filename_pattern = re.compile(rf"^{re.escape(student_id)}_(\d{{2}})\.wav$")
expected_numbers = set(range(1, EXPECTED_FILES + 1))

errors = []
warnings = []

if not os.path.exists(target_dir):
    errors.append(f"提出フォルダが見つかりません: {target_dir}")
    errors.append("Google Drive 上の保存先と student_id を確認してください。")
elif not os.path.isdir(target_dir):
    errors.append(f"提出先がフォルダではありません: {target_dir}")
else:
    entries = sorted(os.listdir(target_dir))
    wav_files = [f for f in entries if f.lower().endswith('.wav')]
    non_wav_files = [f for f in entries if not f.lower().endswith('.wav')]

    if non_wav_files:
        warnings.append(f"WAV 以外のファイルがあります: {', '.join(non_wav_files[:10])}")
        if len(non_wav_files) > 10:
            warnings.append(f"他にも {len(non_wav_files) - 10} 件あります。")

    if len(wav_files) != EXPECTED_FILES:
        errors.append(f"WAV ファイル数が {len(wav_files)} 個です（期待値: {EXPECTED_FILES} 個）")

    seen_numbers = []
    invalid_name_files = []

    for fname in wav_files:
        fpath = os.path.join(target_dir, fname)
        m = filename_pattern.match(fname)
        if not m:
            invalid_name_files.append(fname)
        else:
            num = int(m.group(1))
            seen_numbers.append(num)
            if num not in expected_numbers:
                errors.append(f"{fname}: 連番が範囲外です（01〜20 の範囲である必要があります）")

        try:
            with wave.open(fpath, 'rb') as wf:
                n_channels = wf.getnchannels()
                framerate = wf.getframerate()
                n_frames = wf.getnframes()
                duration = n_frames / float(framerate) if framerate > 0 else 0.0

                if framerate != REQUIRED_SR:
                    errors.append(f"{fname}: サンプリングレートが {framerate} Hz です（期待値: {REQUIRED_SR} Hz）")

                if n_channels != REQUIRED_CHANNELS:
                    errors.append(f"{fname}: チャンネル数が {n_channels} です（期待値: {REQUIRED_CHANNELS}）")

                if duration > MAX_DURATION:
                    errors.append(f"{fname}: 長さが {duration:.2f} 秒です（最大 {MAX_DURATION} 秒以内にしてください）")

        except (wave.Error, EOFError):
            errors.append(f"{fname}: wav ファイルとして読み込めません")

    if invalid_name_files:
        errors.append("命名規則に合わないファイルがあります: " + ', '.join(invalid_name_files[:10]))
        if len(invalid_name_files) > 10:
            errors.append(f"他にも {len(invalid_name_files) - 10} 件あります。")

    counts = Counter(seen_numbers)
    duplicates = sorted(num for num, count in counts.items() if count > 1)
    missing = sorted(expected_numbers - set(seen_numbers))

    if duplicates:
        errors.append("重複している連番があります: " + ', '.join(f'{num:02d}' for num in duplicates))
    if missing:
        errors.append("不足している連番があります: " + ', '.join(f'{num:02d}' for num in missing))

print(f'チェック対象: {target_dir}')
if errors:
    print('録音データに問題があります。以下を修正してください。')
    for e in errors:
        print(' -', e)
else:
    print('すべての必須条件を満たしています。提出可能です。')

if warnings:
    print('\n参考情報:')
    for w in warnings:
        print(' -', w)
